<a href="https://colab.research.google.com/github/ritashreemukherjee123/Project-1-Applied-Search-Intelligence-Google-Search-Ranking-Discoverability.-/blob/main/work/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Search Intelligence data Contract - An active, provable commitment written before any ML modeling.**

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one client page, `fact_content_query_90d` means client x content x query hash (fixed 90-day window) in the warehouse data.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('RitashreeM@123')

RitashreeM@123··········


**Important**: That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

In [3]:
##Connecting DuckDB to the release which would autheticate every query written
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')



dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [12]:
#Join content facts to dim_content on content_hash_id.
result = con.sql(f"""
SELECT
    t1.client_hash_id,
    t1.content_hash_id AS fact_content_hash_id,
    t1.query_hash_id,
    t2.content_hash_id AS dim_content_hash_id,
    t2.content_type,
    t2.content_created_date,
    t2.content_updated_date,
    t1.query_char_count
FROM
    {TABLES['fact_query_90d']} AS t1
JOIN
    {TABLES['dim_content']} AS t2
ON
    t1.content_hash_id = t2.content_hash_id
LIMIT 5;
""").fetchdf()
print(result)

            client_hash_id      fact_content_hash_id           query_hash_id  \
0  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_58b1b001f839d699   
1  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_922b8eca2a24cd34   
2  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_9f0c36a6ae2a6a99   
3  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_a032820b5467e996   
4  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_ba1a2f131961c5da   

        dim_content_hash_id     content_type content_created_date  \
0  content_447894f2faf0d2bc  keyword article           2025-08-04   
1  content_447894f2faf0d2bc  keyword article           2025-08-04   
2  content_447894f2faf0d2bc  keyword article           2025-08-04   
3  content_447894f2faf0d2bc  keyword article           2025-08-04   
4  content_447894f2faf0d2bc  keyword article           2025-08-04   

  content_updated_date  query_char_count  
0           2026-06-10                17  
1           2026-0

In [15]:
# Join client-level facts to dim_clients on client_hash_id.
result_client_join = con.sql(f"""
SELECT
    t1.client_hash_id AS fact_client_hash_id,
    t1.content_hash_id,
    t1.query_hash_id,
    t2.client_hash_id AS dim_client_hash_id,
    t2.client_created_date,
    t1.query_char_count
FROM
    {TABLES['fact_query_90d']} AS t1
JOIN
    {TABLES['dim_clients']} AS t2
ON
    t1.client_hash_id = t2.client_hash_id
LIMIT 5;
""").fetchdf()
print(result_client_join)

       fact_client_hash_id           content_hash_id           query_hash_id  \
0  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_58b1b001f839d699   
1  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_922b8eca2a24cd34   
2  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_9f0c36a6ae2a6a99   
3  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_a032820b5467e996   
4  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_ba1a2f131961c5da   

        dim_client_hash_id client_created_date  query_char_count  
0  client_08a6a72ff48e62c0          2025-05-26                17  
1  client_08a6a72ff48e62c0          2025-05-26                34  
2  client_08a6a72ff48e62c0          2025-05-26                16  
3  client_08a6a72ff48e62c0          2025-05-26                24  
4  client_08a6a72ff48e62c0          2025-05-26                18  


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



In [16]:
content_analysis_result = con.sql(f"""
SELECT
    t1.client_hash_id,
    t1.content_hash_id AS fact_content_hash_id,
    t1.query_hash_id,
    t2.content_hash_id AS dim_content_hash_id,
    t2.content_type,
    t2.content_created_date,
    t2.content_updated_date,
    t2.keyword_hash_id,
    t2.url_hash_id,
    t1.query_char_count
FROM
    {TABLES['fact_query_90d']} AS t1
JOIN
    {TABLES['dim_content']} AS t2
ON
    t1.content_hash_id = t2.content_hash_id
LIMIT 5;
""").fetchdf()
print(content_analysis_result)

            client_hash_id      fact_content_hash_id           query_hash_id  \
0  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_58b1b001f839d699   
1  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_922b8eca2a24cd34   
2  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_9f0c36a6ae2a6a99   
3  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_a032820b5467e996   
4  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_ba1a2f131961c5da   

        dim_content_hash_id     content_type content_created_date  \
0  content_447894f2faf0d2bc  keyword article           2025-08-04   
1  content_447894f2faf0d2bc  keyword article           2025-08-04   
2  content_447894f2faf0d2bc  keyword article           2025-08-04   
3  content_447894f2faf0d2bc  keyword article           2025-08-04   
4  content_447894f2faf0d2bc  keyword article           2025-08-04   

  content_updated_date           keyword_hash_id           url_hash_id  \
0           2026-06-10  keywor

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.